## Example 1: The command-line parser and the JSON listing

In [ ]:
import json
from piplicenses import create_output_string, create_parser

args = create_parser().parse_args(["--format=json", "--with-system", "--packages", "prettytable", "pip-licenses"])
rows = json.loads(create_output_string(args))
print(json.dumps(rows, indent=1))
assert {row["Name"] for row in rows} == {"prettytable", "pip-licenses"}, rows
assert all(set(row) == {"Name", "Version", "License"} for row in rows), rows

## Example 2: Every license source at once, checked against the installed metadata

In [ ]:
import json
from importlib.metadata import metadata
from piplicenses import create_output_string, create_parser

args = create_parser().parse_args(["--format=json", "--with-system", "--from=all", "--packages", "prettytable"])
(row,) = json.loads(create_output_string(args))
print(row)
expected = metadata("prettytable").get("License-Expression") or "UNKNOWN"
assert row["License-Expression"] == expected, (row, expected)
assert {"License-Metadata", "License-Classifier", "License-Expression"} <= set(row), row
assert row["Version"] == metadata("prettytable")["Version"], row

## Example 3: The license file and its text

In [ ]:
import json
from pathlib import Path
from piplicenses import create_output_string, create_parser

args = create_parser().parse_args(
    ["--format=json", "--with-system", "--with-license-file", "--packages", "pip-licenses"]
)
(row,) = json.loads(create_output_string(args))
license_path = Path(row["LicenseFile"])
print(license_path)
assert license_path.is_file(), row["LicenseFile"]
assert row["LicenseText"] == license_path.read_text(encoding="utf-8"), "the text is not the file's"
assert "MIT" in row["LicenseText"] or "Permission is hereby granted" in row["LicenseText"]

## Example 4: CSV and JSON describe the same packages

In [ ]:
import csv
import io
import json
from piplicenses import create_output_string, create_parser

names = ["prettytable", "wcwidth", "pip-licenses"]
as_json = create_parser().parse_args(["--format=json", "--with-system", "--packages", *names])
as_csv = create_parser().parse_args(["--format=csv", "--with-system", "--packages", *names])
from_json = sorted(json.loads(create_output_string(as_json)), key=lambda row: row["Name"])
from_csv = sorted(csv.DictReader(io.StringIO(create_output_string(as_csv))), key=lambda row: row["Name"])
print(from_csv)
assert from_csv == from_json, (from_csv, from_json)
assert len(from_json) == 3, from_json

## Example 5: The exit codes a gate relies on, and reading another interpreter

In [ ]:
import json
import subprocess
import sys

base = [sys.executable, "-m", "piplicenses", "--with-system", "--packages", "prettytable"]
failed = subprocess.run([*base, "--fail-on=BSD-3-Clause"], capture_output=True, text=True)
print(failed.returncode, failed.stderr)
assert failed.returncode == 1 and "prettytable" in failed.stderr, failed
refused = subprocess.run([*base, "--allow-only=MIT"], capture_output=True, text=True)
assert refused.returncode == 1 and "not in allow-only" in refused.stderr, refused
read = subprocess.run([*base, "--format=json", f"--python={sys.executable}"], capture_output=True, text=True)
assert read.returncode == 0, read.stderr
assert [row["Name"] for row in json.loads(read.stdout)] == ["prettytable"], read.stdout